In [0]:
%sql
CREATE DATABASE IF NOT EXISTS construction_kpi;

In [0]:
# Show current catalog / schema / database
print("Current database:", spark.catalog.currentDatabase())

# Try to create/use your database
spark.sql("CREATE DATABASE IF NOT EXISTS construction_kpi")
spark.sql("USE construction_kpi")

# Confirm
db = spark.sql("SELECT current_database()").collect()[0][0]
print("Now using database:", db)

In [0]:
%sql
SHOW TABLES IN construction_kpi;

###Bronze Table Validation

In [0]:
%sql
SELECT 'bronze_projects' AS table_name, COUNT(*) AS rows FROM construction_kpi.bronze_projects
UNION ALL
SELECT 'bronze_cost_actuals', COUNT(*) FROM construction_kpi.bronze_cost_actuals
UNION ALL
SELECT 'bronze_schedule_updates', COUNT(*) FROM construction_kpi.bronze_schedule_updates
UNION ALL
SELECT 'bronze_change_orders', COUNT(*) FROM construction_kpi.bronze_change_orders;

###Create Silver Projects table

In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.silver_projects AS
SELECT
  TRIM(project_id)                   AS project_id,
  TRIM(project_name)                 AS project_name,
  TRIM(client_name)                  AS client_name,
  TRIM(region)                       AS region,
  TRIM(project_type)                 AS project_type,
  CAST(start_date AS DATE)           AS start_date,
  CAST(baseline_finish_date AS DATE) AS baseline_finish_date,
  CAST(baseline_budget AS DOUBLE)    AS baseline_budget,
  TRIM(project_manager)              AS project_manager
FROM construction_kpi.bronze_projects
WHERE project_id IS NOT NULL;

In [0]:
%sql
SELECT COUNT(*) AS silver_projects_rows
FROM construction_kpi.silver_projects;

In [0]:
%sql
SELECT *
FROM construction_kpi.silver_projects
LIMIT 5;

####Create silver_cost_actuals

In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.silver_cost_actuals AS
SELECT
  TRIM(project_id)              AS project_id,
  CAST(cost_date AS DATE)       AS cost_date,
  TRIM(cost_category)           AS cost_category,
  TRIM(vendor)                  AS vendor,
  CAST(actual_cost AS DOUBLE)   AS actual_cost
FROM construction_kpi.bronze_cost_actuals
WHERE project_id IS NOT NULL
  AND actual_cost IS NOT NULL
  AND CAST(actual_cost AS DOUBLE) >= 0;

In [0]:
%sql
SELECT COUNT(*) AS silver_cost_actuals_rows
FROM construction_kpi.silver_cost_actuals;

In [0]:
%sql
SELECT *
FROM construction_kpi.silver_cost_actuals
LIMIT 5;

####Create silver_schedule_updates


In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.silver_schedule_updates AS
SELECT
  TRIM(project_id)                 AS project_id,
  CAST(update_date AS DATE)        AS update_date,
  CAST(percent_complete AS DOUBLE) AS percent_complete,
  CAST(current_finish_date AS DATE) AS current_finish_date,
  CAST(delay_days AS INT)          AS delay_days
FROM construction_kpi.bronze_schedule_updates
WHERE project_id IS NOT NULL
  AND percent_complete IS NOT NULL
  AND CAST(percent_complete AS DOUBLE) BETWEEN 0 AND 100;

In [0]:
%sql
SELECT COUNT(*) AS silver_schedule_updates_rows
FROM construction_kpi.silver_schedule_updates;

In [0]:
%sql
SELECT *
FROM construction_kpi.silver_schedule_updates
ORDER BY project_id, update_date
LIMIT 8;

####Create silver_change_orders

In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.silver_change_orders AS
SELECT
  TRIM(project_id)               AS project_id,
  TRIM(co_id)                    AS co_id,
  CAST(co_date AS DATE)          AS co_date,
  UPPER(TRIM(co_status))         AS co_status,
  CAST(co_amount AS DOUBLE)      AS co_amount,
  TRIM(reason)                   AS reason
FROM construction_kpi.bronze_change_orders
WHERE project_id IS NOT NULL
  AND co_amount IS NOT NULL
  AND CAST(co_amount AS DOUBLE) >= 0;

In [0]:
%sql
SELECT COUNT(*) AS silver_change_orders_rows
FROM construction_kpi.silver_change_orders;

In [0]:
%sql
SELECT *
FROM construction_kpi.silver_change_orders
ORDER BY project_id, co_date
LIMIT 8;

####create gold_project_kpis

In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.gold_project_kpis AS
WITH actual_cost AS (
  SELECT
    project_id,
    SUM(actual_cost) AS total_actual_cost
  FROM construction_kpi.silver_cost_actuals
  GROUP BY project_id
),
approved_cos AS (
  SELECT
    project_id,
    SUM(co_amount) AS approved_change_orders_amount
  FROM construction_kpi.silver_change_orders
  WHERE co_status = 'APPROVED'
  GROUP BY project_id
),
latest_schedule AS (
  SELECT
    project_id,
    update_date,
    percent_complete,
    delay_days,
    current_finish_date,
    ROW_NUMBER() OVER (PARTITION BY project_id ORDER BY update_date DESC) AS rn
  FROM construction_kpi.silver_schedule_updates
)
SELECT
  p.project_id,
  p.project_name,
  p.client_name,
  p.region,
  p.project_type,
  p.baseline_budget,

  COALESCE(a.total_actual_cost, 0) AS total_actual_cost,
  COALESCE(c.approved_change_orders_amount, 0) AS approved_change_orders_amount,

  (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)) AS total_forecast_budget,

  (COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))) AS cost_variance_amount,

  CASE
    WHEN (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)) = 0 THEN NULL
    ELSE (COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)))
         / (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))
  END AS cost_variance_pct,

  s.percent_complete AS latest_percent_complete,
  s.delay_days       AS latest_delay_days,
  s.current_finish_date AS latest_finish_date

FROM construction_kpi.silver_projects p
LEFT JOIN actual_cost a
  ON p.project_id = a.project_id
LEFT JOIN approved_cos c
  ON p.project_id = c.project_id
LEFT JOIN latest_schedule s
  ON p.project_id = s.project_id AND s.rn = 1;

In [0]:
%sql
SELECT COUNT(*) AS gold_project_kpis_rows
FROM construction_kpi.gold_project_kpis;

In [0]:
%sql
SELECT
  project_id, project_name, baseline_budget, total_actual_cost, approved_change_orders_amount,
  total_forecast_budget, cost_variance_amount, cost_variance_pct, latest_percent_complete, latest_delay_days
FROM construction_kpi.gold_project_kpis
ORDER BY cost_variance_pct DESC
LIMIT 12;

In [0]:
%sql
CREATE OR REPLACE TABLE construction_kpi.gold_project_kpis AS
WITH actual_cost AS (
  SELECT
    project_id,
    SUM(actual_cost) AS total_actual_cost
  FROM construction_kpi.silver_cost_actuals
  GROUP BY project_id
),
approved_cos AS (
  SELECT
    project_id,
    SUM(co_amount) AS approved_change_orders_amount
  FROM construction_kpi.silver_change_orders
  WHERE co_status = 'APPROVED'
  GROUP BY project_id
),
latest_schedule AS (
  SELECT
    project_id,
    update_date,
    percent_complete,
    delay_days,
    current_finish_date,
    ROW_NUMBER() OVER (PARTITION BY project_id ORDER BY update_date DESC) AS rn
  FROM construction_kpi.silver_schedule_updates
)
SELECT
  p.project_id,
  p.project_name,
  p.client_name,
  p.region,
  p.project_type,
  p.baseline_budget,

  COALESCE(a.total_actual_cost, 0) AS total_actual_cost,
  COALESCE(c.approved_change_orders_amount, 0) AS approved_change_orders_amount,

  (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)) AS total_forecast_budget,

  (COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))) AS cost_variance_amount,

  CASE
    WHEN (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)) = 0 THEN NULL
    ELSE (COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)))
         / (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))
  END AS cost_variance_pct,

  s.percent_complete AS latest_percent_complete,
  s.delay_days       AS latest_delay_days,
  s.current_finish_date AS latest_finish_date,

  CASE
    WHEN 
      ((COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)))
        / (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))) > 0.15
      OR s.delay_days > 30
      THEN 'RED'

    WHEN 
      ((COALESCE(a.total_actual_cost, 0) - (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0)))
        / (p.baseline_budget + COALESCE(c.approved_change_orders_amount, 0))) BETWEEN 0.05 AND 0.15
      OR s.delay_days BETWEEN 10 AND 30
      THEN 'AMBER'

    ELSE 'GREEN'
  END AS risk_status

FROM construction_kpi.silver_projects p
LEFT JOIN actual_cost a
  ON p.project_id = a.project_id
LEFT JOIN approved_cos c
  ON p.project_id = c.project_id
LEFT JOIN latest_schedule s
  ON p.project_id = s.project_id AND s.rn = 1;

In [0]:
%sql
SELECT risk_status, COUNT(*) 
FROM construction_kpi.gold_project_kpis
GROUP BY risk_status;

In [0]:
%sql
SELECT project_id, project_name, cost_variance_pct, latest_delay_days, risk_status
FROM construction_kpi.gold_project_kpis
ORDER BY risk_status DESC, cost_variance_pct DESC;